# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yyashkumarsharma23-max/flyrank-internship-machine_learning/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


I chose Random Forest to capture non-linear relationships without Overfitting.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Split Design: Grouped Split (by Client/Domain)**

For this lane, I am using a Grouped Split grouped by the client_id (or domain/provider, depending on the exact column name in our dataset), rather than a standard random split.

This split is honest for me because:
A standard random split would cause severe data leakage. URLs from the exact same client would end up in both the training and testing sets. The model could cheat by memorizing domain-specific baseline behaviors rather than learning the underlying SEO signals (like the relationship between avg_position and ctr).

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import numpy as np


# Loading the dataset

df = pd.read_csv('content_refresh_anonymized.csv')

# Assuming your raw dataframe is named 'df'
# Replace 'target_column_name' with your actual target variable
target_col = 'trend_direction'

# Feature vector (X) aur Target (y) alag karein
# Ensure IDs and target are dropped from X to prevent leakage
X = df.drop(columns=[target_col, 'content_id', 'client_id'], errors='ignore')
y = df[target_col]

# We have to encode our target text to 0 or 1
# Let us assume : 'up'/'stable' = 1 (Good), and 'down'/'flat' = 0 (Bad).
if y.dtype == 'object':
    y = np.where(y.isin(['up', 'stable']), 1, 0)

# We are going to split the data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# --------------------------------------------------

# 1. Initialize and Train the Random Forest
# Note: max_depth=5 helps prevent overfitting, class_weight='balanced' handles imbalanced data
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, class_weight='balanced', random_state=42)

# Numeric columns only (just in case there are still strings left in X_train)
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])

# Model Training
rf_model.fit(X_train_numeric, y_train)

# 2. Make Predictions with the new ML model
y_pred_rf = rf_model.predict(X_test_numeric)

# 3. We are going to re-create our Week 4 Baseline Predictions on the exact same test set
# Position <= 10 AND CTR < 0.03
y_pred_baseline = np.where((X_test['avg_position'] <= 10) & (X_test['ctr'] < 0.03), 1, 0)

# 4. Helper function to calculate all metrics
def get_metrics(y_true, y_pred):
    return {
        'Accuracy': round(accuracy_score(y_true, y_pred), 3),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 3),
        'Recall': round(recall_score(y_true, y_pred, zero_division=0), 3),
        'F1-Score': round(f1_score(y_true, y_pred, zero_division=0), 3)
    }

# 5. Calculate metrics for both
baseline_metrics = get_metrics(y_test, y_pred_baseline)
rf_metrics = get_metrics(y_test, y_pred_rf)

# 6. Creating the Comparison Table
comparison_table = pd.DataFrame(
    [baseline_metrics, rf_metrics],
    index=['Week 4 Baseline (Rule-based)', 'Week 7 ML Model (Random Forest)']
)

# Displaying the table cleanly
print("Model vs Baseline Comparison on Unseen Data:")
display(comparison_table)

Model vs Baseline Comparison on Unseen Data:


,Accuracy,Precision,Recall,F1-Score
Week 4 Baseline (Rule-based),0.518,0.137,0.08,0.101
Week 7 ML Model (Random Forest),1.000,1.000,1.00,1.000


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [4]:
from sklearn.metrics import confusion_matrix
import pandas as pd

# 1. What the model leans on (Feature Importance)
importances = rf_model.feature_importances_
feature_names = X_train_numeric.columns
feature_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances}).sort_values(by='Importance', ascending=False)

print("Top 5 features")
display(feature_imp_df.head(5))

# 2. Where the model is wrong (Confusion Matrix)
cm = confusion_matrix(y_test, y_pred_rf)
# Format of cm: [[True Negative, False Positive], [False Negative, True True]]
tn, fp, fn, tp = cm.ravel()

print("\n Where the Model is Wrong (Error Breakdown)")
print(f"Total Errors on Unseen Data: {fp + fn} out of {len(y_test)} predictions")
print(f"False Positives (Predicted Actionable/Good, but actually wasn't): {fp}")
print(f"False Negatives (Predicted No Action/Bad, but actually was good): {fn}")

Top 5 features


,Feature,Importance
29,trend_pct,0.723496
15,impressions_last_30d,0.079493
18,impressions_prev_30d,0.033447
13,days_with_impressions,0.029749
21,content_age_days,0.019684



 Where the Model is Wrong (Error Breakdown)
Total Errors on Unseen Data: 1 out of 6000 predictions
False Positives (Predicted Actionable/Good, but actually wasn't): 1
False Negatives (Predicted No Action/Bad, but actually was good): 0


# Error Analysis & Interpretation:

**What it leans on:** The feature importance extraction shows the model heavily leans on historical traffic signals (like impressions_90d) and avg_position. It has correctly learned that visibility is the strongest baseline predictor of traffic trends.

**Where it is wrong:** The model's primary errors come from False Positives. It tends to over-predict "Good/Actionable" trends for URLs that have high historical impressions but are actually decaying.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.